# CogVideoX Real Cartoon Video (Free Colab Quickstart)

This notebook generates a **real motion text-to-video** clip using CogVideoX.

- No Pollinations key needed
- No slideshow pipeline
- Requires Colab GPU runtime


## 1) Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece safetensors torch

## 2) Generate Real Motion Video

In [ ]:
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

# Better quality usually needs more VRAM/time.
MODEL_ID = "THUDM/CogVideoX-2b"
PROMPT = """
A colorful kids cartoon animation of two friends exploring a magical forest,
expressive character movement, cinematic camera motion, warm lighting,
storybook style, smooth motion, child-safe.
"""

pipe = CogVideoXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
)
pipe.enable_model_cpu_offload()
if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()

result = pipe(
    prompt=PROMPT,
    num_videos_per_prompt=1,
    num_inference_steps=30,
    num_frames=49,
    guidance_scale=6,
)

frames = result.frames[0]
export_to_video(frames, "cartoon_video.mp4", fps=8)
print("Saved: cartoon_video.mp4")

## 3) Download Video

In [ ]:
from google.colab import files
files.download("cartoon_video.mp4")

## Optional: Call Your Backend API (`engine: cogvideox`)

Use this only if your backend runtime has GPU + model dependencies installed.

In [ ]:
import requests

API_BASE = "https://your-backend-domain.com"

payload = {
    "prompt": "A kids cartoon where two friends solve a mystery in a glowing forest.",
    "engine": "cogvideox",
    "videoSize": "youtube",
    "numFrames": 49,
    "numInferenceSteps": 30,
    "guidanceScale": 6,
    "language": "en"
}

response = requests.post(f"{API_BASE}/api/kids-video-hf/generate", json=payload, timeout=1800)
print(response.status_code)
print(response.json())